In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm 

Literature:
Efficient Market Hypothesis (EMH)
1. Moskowitz, Ooi & Pedersen (2012) — *Time Series Momentum* — main reference. Read Abstract → Introduction → Conclusion first; flag unfamiliar formulas for later review.
2. *A Century of Evidence on Trend-Following Investing*


## Real Data: S&P 500 Total Return Index

#### Data source
The S&P 500 Total Return series was originally downloaded with `yfinance` using ticker `^SP500TR` and saved locally as `sp500tr_daily_1988_2026-07.csv`.


In [2]:
df_sp500 = pd.read_csv("sp500tr_daily_1988_2026-07.csv",
                      index_col="Date",
                    parse_dates=["Date"])

In [3]:
df_sp500.index = pd.to_datetime(df_sp500.index, utc=True)

In [4]:
df_sp500.shape

(9717, 7)

In [5]:
df_sp500.index = df_sp500.index.tz_localize(None)

In [6]:
df_sp500.isna().sum()

Open            0
High            0
Low             0
Close           0
Volume          0
Dividends       0
Stock Splits    0
dtype: int64

In [7]:
df_sp500.index.is_monotonic_increasing

True

In [8]:
df_sp500.index.duplicated().sum()

np.int64(0)

In [9]:
df_sp500 = df_sp500.resample('ME').last()

In [10]:
df_sp500 = df_sp500.drop(columns=["Open", "High", "Low", "Volume", "Dividends", "Stock Splits"])

In [11]:
df_sp500["return_12m"] = df_sp500["Close"]/df_sp500["Close"].shift(12)-1

In [12]:
df_sp500["future_return_3m"] = df_sp500["Close"].shift(-3)/df_sp500["Close"]-1

In [13]:
df_sp500.isna().sum()

Close                0
return_12m          12
future_return_3m     3
dtype: int64

#### Positive 12-month signal


In [14]:
mean_3m = df_sp500[df_sp500["return_12m"] > 0]["future_return_3m"].mean()
mean_3m

np.float64(0.03373791113003564)

In [15]:
probability_3m = (df_sp500[df_sp500["return_12m"] > 0]["future_return_3m"] > 0).mean()
probability_3m

np.float64(0.7480106100795756)

#### Benchmark (unconditional market return)


In [16]:
mean_3m_benchmark = df_sp500["future_return_3m"].mean()
mean_3m_benchmark

np.float64(0.03008458710698316)

In [17]:
probability_3m_benchmark = (df_sp500["future_return_3m"] > 0).mean()
probability_3m_benchmark

np.float64(0.7257019438444925)

#### Clean analysis sample


In [18]:
analysis_df = df_sp500[
    ["Close", "return_12m", "future_return_3m"]
].dropna().copy()

In [19]:
mean_3m = analysis_df[analysis_df["return_12m"] > 0]["future_return_3m"].mean()
mean_3m

np.float64(0.03373791113003564)

In [20]:
probability_3m = (analysis_df[analysis_df["return_12m"] > 0]["future_return_3m"] > 0).mean()
probability_3m

np.float64(0.7540106951871658)

In [21]:
mean_3m_benchmark = analysis_df["future_return_3m"].mean()
mean_3m_benchmark

np.float64(0.029828444623906606)

In [22]:
probability_3m_benchmark = (analysis_df["future_return_3m"] > 0).mean()
probability_3m_benchmark

np.float64(0.7254464285714286)

#### Non-positive 12-month signal


In [23]:
mean_3m_neg = analysis_df[analysis_df["return_12m"] <= 0]["future_return_3m"].mean()
mean_3m_neg

np.float64(0.010069789579416637)

In [24]:
probability_3m_neg = (analysis_df[analysis_df["return_12m"] <= 0]["future_return_3m"] > 0).mean()
probability_3m_neg

np.float64(0.581081081081081)

In [25]:
[(analysis_df["return_12m"] <= 0).sum(), 
(analysis_df["return_12m"] > 0).sum()]

[np.int64(74), np.int64(374)]

In [26]:
df_sp500["future_return_1m"] = df_sp500["Close"].shift(-1)/df_sp500["Close"]-1

In [27]:
analysis_1m_df = df_sp500[
    ["Close", "return_12m", "future_return_1m"]
].dropna().copy()

In [28]:
mean_1m = analysis_1m_df[analysis_1m_df["return_12m"] > 0]["future_return_1m"].mean()
mean_1m

np.float64(0.011167976212077584)

In [29]:
probability_1m = (analysis_1m_df[analysis_1m_df["return_12m"] > 0]["future_return_1m"] > 0).mean()
probability_1m

np.float64(0.6781914893617021)

In [30]:
mean_1m_benchmark = analysis_1m_df["future_return_1m"].mean()
mean_1m_benchmark

np.float64(0.00981704247330504)

In [31]:
probability_1m_benchmark = (analysis_1m_df["future_return_1m"] > 0).mean()
probability_1m_benchmark

np.float64(0.6577777777777778)

In [32]:
mean_1m_neg = analysis_1m_df[analysis_1m_df["return_12m"] <= 0]["future_return_1m"].mean()
probability_1m_neg = (analysis_1m_df[analysis_1m_df["return_12m"] <= 0]["future_return_1m"] > 0).mean()
[mean_1m_neg, probability_1m_neg]

[np.float64(0.00295283861143375), np.float64(0.5540540540540541)]

#### Hypotheses for the 1-month forward return (leaving the 3-month horizon aside for now)

**H0:** The mean return over the next month does not depend on whether the previous 12-month return was positive or non-positive.

**H1:** The mean return over the next month is higher after a positive 12-month return.

The observed means are 1.12% after positive 12-month returns and 0.30% after non-positive 12-month returns. A difference of 0.82 percentage points by itself is not enough to reject $H_0$; we also need to account for dispersion and sample size.

For this we need:
- difference in means
- dispersion / standard deviation
- sample size


In [33]:
std_1m_positive_12m = (analysis_1m_df[analysis_1m_df["return_12m"] > 0]["future_return_1m"]).std()
std_1m_nonpositive_12m = (analysis_1m_df[analysis_1m_df["return_12m"] <= 0]["future_return_1m"]).std()
[std_1m_positive_12m, std_1m_nonpositive_12m]

[np.float64(0.03744402192903582), np.float64(0.061673804185283225)]

In [34]:
# Number of observations
n_1m_positive_12m = (analysis_1m_df[analysis_1m_df["return_12m"] > 0]["future_return_1m"]).count()
n_1m_nonpositive_12m = (analysis_1m_df[analysis_1m_df["return_12m"] <= 0]["future_return_1m"]).count()
[n_1m_positive_12m, n_1m_nonpositive_12m]


[np.int64(376), np.int64(74)]

In [35]:
se_1m_positive_12m = (std_1m_positive_12m)/np.sqrt(n_1m_positive_12m)
se_1m_nonpositive_12m = (std_1m_nonpositive_12m)/np.sqrt(n_1m_nonpositive_12m)
[se_1m_positive_12m, se_1m_nonpositive_12m]

[np.float64(0.0019310279881493858), np.float64(0.007169434108887854)]

#### Standard error of the difference in means


In [36]:
se_diff_1m = np.sqrt((se_1m_positive_12m**2)+(se_1m_nonpositive_12m**2))
se_diff_1m

np.float64(0.007424934648379126)

How far is the estimated effect from zero?


In [37]:
t_statistic = (mean_1m - mean_1m_neg)/se_diff_1m
t_statistic

np.float64(1.106425576747293)

#### If $H_0$ is true, what is the probability of observing a result at least as extreme as ours?


A conventional significance level is:
$$ \alpha=0.05 $$

Decision rule:
$$ p<0.05\Rightarrow\text{reject }H_0 $$
$$ p\geq0.05\Rightarrow\text{fail to reject }H_0 $$


In [38]:
from scipy import stats 

#### Welch–Satterthwaite degrees-of-freedom formula
$$
df =
\frac{
\left(
\frac{s_+^2}{n_+} + \frac{s_-^2}{n_-}
\right)^2
}{
\frac{\left(\frac{s_+^2}{n_+}\right)^2}{n_+ - 1}
+
\frac{\left(\frac{s_-^2}{n_-}\right)^2}{n_- - 1}
}
$$


In [39]:
df_welch = (
    (std_1m_positive_12m**2 / n_1m_positive_12m + std_1m_nonpositive_12m**2 / n_1m_nonpositive_12m)**2
    / (
        (std_1m_positive_12m**2 / n_1m_positive_12m)**2 / (n_1m_positive_12m - 1)
        + (std_1m_nonpositive_12m**2 / n_1m_nonpositive_12m)**2 / (n_1m_nonpositive_12m - 1)
    )
)

p_value_1m = stats.t.sf(t_statistic, df=df_welch)
p_value_1m


np.float64(0.1358525508457939)

In [40]:
p_value_1m = stats.t.sf(t_statistic, df=df_welch)
p_value_1m


np.float64(0.1358525508457939)

In [41]:
positive_1m = analysis_1m_df.loc[
    analysis_1m_df["return_12m"] > 0,
    "future_return_1m"
]

nonpositive_1m = analysis_1m_df.loc[
    analysis_1m_df["return_12m"] <= 0,
    "future_return_1m"
]

In [42]:
stats.ttest_ind(
    positive_1m,
    nonpositive_1m,
    equal_var=False,
    alternative="greater"
)

TtestResult(statistic=np.float64(1.106425576747293), pvalue=np.float64(0.1358525508457939), df=np.float64(83.88980601834777))

#### Are the observations independent? Time dependence


Check autocorrelation:
1. `future_return_1m` | $Corr(R_{t+1},R_t)$
2. `return_12m > 0` (binary signal) | $Corr(S_t,S_{t-1})$


In [43]:
# Binary signal
analysis_1m_df["signal"] = (analysis_1m_df["return_12m"] > 0).astype(int)


In [44]:
# Autocorrelation
return_autocorr_1 = analysis_1m_df["future_return_1m"].autocorr(lag=1)

signal_autocorr_1 = analysis_1m_df["signal"].autocorr(lag=1)


In [45]:
return_autocorr_1, signal_autocorr_1


(np.float64(-0.00984613590093637), np.float64(0.8381981981981981))

#### Regression and residual diagnostics


$$
Y_t = R_{t+1}
$$

$$
X_t = S_t
$$

In [46]:
y = analysis_1m_df["future_return_1m"]
x = analysis_1m_df["signal"]

### Regression without a constant


If only the signal is passed to the model:

$$
R_{t+1} = \beta S_t + \varepsilon_{t+1}
$$

When $S_t = 0$:

$$
\hat{R}_{t+1} = \beta \cdot 0 = 0
$$

Therefore, without a constant the model is forced to predict zero expected return when the signal equals zero.

### Regression with a constant

After adding a constant:

$$
R_{t+1} = \alpha + \beta S_t + \varepsilon_{t+1}
$$

Equivalently:

$$
R_{t+1} = \alpha \cdot 1 + \beta S_t + \varepsilon_{t+1}
$$

When $S_t = 0$:

$$
E[R_{t+1} \mid S_t = 0] = \alpha
$$

When $S_t = 1$:

$$
E[R_{t+1} \mid S_t = 1] = \alpha + \beta
$$

Therefore:

$$
\alpha = \mu_-
$$

and

$$
\beta = \mu_+ - \mu_-
$$

In [47]:
import statsmodels.api as sm

X = sm.add_constant(x)


In [48]:
model = sm.OLS(y, X)

In [49]:
results = model.fit()

In [50]:
results.resid.autocorr(lag=1)

np.float64(-0.026931988852050323)

In [51]:
for lag in range(1, 7):
    print(lag)
    print(results.resid.autocorr(lag=lag))


1
-0.026931988852050323
2
-0.07400744122159564
3
0.036410627624712394
4
-0.00024490120627932045
5
0.004054494827118771
6
-0.0997460933013312


In [52]:
sm.stats.diagnostic.acorr_ljungbox(results.resid)

,lb_stat,lb_pvalue
1,0.327961,0.566862
2,2.808270,0.245579
3,3.407492,0.332960
4,3.407516,0.492079
5,3.414865,0.636307
6,7.827498,0.251016
7,9.603523,0.212177
8,11.604539,0.169740
9,13.971696,0.123336
10,13.972522,0.174249


The p-values are above the significance threshold, so I do not find statistically significant evidence of residual autocorrelation.


#### Volatility and heteroskedasticity


#### Heteroskedasticity means that the variance of the errors is not constant
$$
\operatorname{Var}(\varepsilon_t) \neq \sigma^2 \quad \text{for all } t
$$

#### Volatility clustering
Periods of high volatility tend to occur near other high-volatility periods, while low-volatility periods tend to cluster together.


Squaring residuals removes the sign and keeps the scale of the error.


In [53]:
((results.resid)**2).autocorr(lag=1)

np.float64(0.2620431693832626)

In [54]:
sm.stats.diagnostic.acorr_ljungbox(results.resid**2)

,lb_stat,lb_pvalue
1,31.084472,2.470397e-08
2,36.908443,9.670155e-09
3,46.678398,4.068502e-10
4,53.960353,5.364245e-11
5,59.955963,1.241194e-11
6,65.230101,3.871553e-12
7,66.961324,6.061594e-12
8,67.052873,1.892650e-11
9,68.088926,3.604046e-11
10,69.555750,5.401526e-11


In [55]:
# Re-estimate SE, t-statistics, and p-values using HAC
results_hac = model.fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 6}
)


In [56]:
for maxlag in 1, 3, 6, 12:
    results_hac_temp = model.fit(
        cov_type="HAC",
        cov_kwds={"maxlags": maxlag}
    )

    print(maxlag, results_hac_temp.bse["signal"], results_hac_temp.tvalues["signal"], results_hac_temp.pvalues["signal"])

1 0.007846762609811061 1.0469461113010075 0.29512443304713787
3 0.007874523345606461 1.0432552219465345 0.29683010381312347
6 0.008163599966545471 1.006313101365766 0.31426498017950266
12 0.00745987189936312 1.1012437896346714 0.27079056802605206


#### Return to the 3-month horizon


In [57]:
for lag in range(1, 7):
    print(analysis_df["future_return_3m"].autocorr(lag=lag))
    

0.6681323292959889
0.33502782533673336
0.03330918592668938
0.019627629865373242
0.024905364215063883
0.028433314605450614


The autocorrelation pattern is consistent with overlapping forward returns.


#### HAC / Newey–West standard errors


In [58]:
analysis_df["signal"] = (analysis_df["return_12m"] > 0).astype(int)
y_3m = analysis_df["future_return_3m"]
x_3m = analysis_df["signal"]
X_3m = sm.add_constant(x_3m)
model_3m = sm.OLS(y_3m, X_3m)

In [59]:
results_3m = model_3m.fit()

In [60]:
results_3m.bse, results_3m.tvalues, results_3m.pvalues

(const     0.008353
 signal    0.009142
 dtype: float64,
 const     1.205570
 signal    2.589004
 dtype: float64,
 const     0.228622
 signal    0.009940
 dtype: float64)

Estimate the regression with HAC standard errors using a 2-lag bandwidth.


In [61]:
# maxlags allows the covariance estimator to account for autocorrelation up to the specified lag.
# Newey–West therefore adjusts the covariance matrix for both heteroskedasticity and serial dependence in the errors.
results_3m_hac = model_3m.fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 2}
)


In [62]:
results_3m_hac.params["signal"], results_3m_hac.bse["signal"], results_3m_hac.tvalues["signal"], results_3m_hac.pvalues["signal"]



(np.float64(0.023668121550619002),
 np.float64(0.019224515258320595),
 np.float64(1.2311426963223513),
 np.float64(0.21826949907083826))

Check sensitivity to 3, 6, and 12 lags.


In [63]:
for maxlag in 3, 6, 12:
    results_3m_hac = model_3m.fit(
    cov_type="HAC",
    cov_kwds={"maxlags": maxlag}
)
    print(maxlag, results_3m_hac.params["signal"], results_3m_hac.bse["signal"], results_3m_hac.tvalues["signal"], results_3m_hac.pvalues["signal"])

3 0.023668121550619002 0.020612361402051915 1.1482489118526173 0.2508658226954359
6 0.023668121550619002 0.022659781104998528 1.0444991256070892 0.29625452353964865
12 0.023668121550619002 0.02134530346685185 1.1088210381911117 0.26750739281968927


### Statistical result


From 1988 to 2026, the average subsequent 3-month S&P 500 Total Return was approximately **2.36 percentage points higher** after a positive 12-month return than after a non-positive one. Naive OLS suggested statistical significance ($p_{2s}\approx0.010$), but overlapping 3-month forward returns create strong mechanical serial correlation. After applying a HAC/Newey–West correction, statistical significance disappears, and the result is robust to reasonable bandwidth choices.

Thus, the sample shows a **positive momentum-like effect**, but the evidence is not strong enough to reject $H_0$ after accounting for serial dependence.


#### Does the signal translate into a trading strategy?


In [64]:
analysis_1m_df["strategy_return"] = analysis_1m_df["future_return_1m"]*analysis_1m_df["signal"]
analysis_1m_df.head(5)

,Close,return_12m,future_return_1m,signal,strategy_return
Date,,,,,
1989-01-31,309.209991,0.200955,-0.024902,1,-0.024902
1989-02-28,301.510010,0.118859,0.023316,1,0.023316
1989-03-31,308.540009,0.181467,0.051922,1,0.051922
1989-04-30,324.559998,0.229208,0.040455,1,0.040455
1989-05-31,337.690002,0.267986,-0.005656,1,-0.005656


#### Compounded wealth


In [65]:
analysis_1m_df["strategy_wealth"] = (analysis_1m_df["strategy_return"]+1).cumprod()
# Cumulative strategy wealth
analysis_1m_df["benchmark_wealth"] = (analysis_1m_df["future_return_1m"]+1).cumprod()
# Cumulative benchmark wealth


### The strategy earns less than the benchmark, but return alone is not enough: risk matters too.


#### $$CAGR\ (Compound\ Annual\ Growth\ Rate)$$
$$
\text{CAGR}
=
\left(
\frac{V_{\text{final}}}{V_{\text{initial}}}
\right)^{\frac{1}{T}}
-1
$$

$$
V_{\text{initial}} = \text{initial portfolio value}
$$

$$
V_{\text{final}} = \text{final portfolio value}
$$

$$
T = \text{number of years}
$$

$$
T = \frac{N}{12}
$$


What constant annual growth rate would produce the same path from $1 to $51.19 over the full sample period?


In [66]:
# Number of years in the sample
T_years = len(analysis_1m_df)/12


In [67]:
strategy_cagr = ((analysis_1m_df["strategy_wealth"].iloc[-1])**(1/T_years))-1
strategy_cagr

np.float64(0.11005521268018903)

In [68]:
benchmark_cagr = ((analysis_1m_df["benchmark_wealth"].iloc[-1])**(1/T_years))-1
benchmark_cagr

np.float64(0.11235371471365663)

#### Annualized volatility


In [69]:
strategy_vol = analysis_1m_df["strategy_return"].std()*np.sqrt(12)
benchmark_vol = analysis_1m_df["future_return_1m"].std()*np.sqrt(12)

The benchmark is more volatile over this sample.


#### Risk-adjusted return (Sharpe ratio)


When the signal is zero, the strategy is allocated to the monthly risk-free rate. 
$$
R_{f,m}
=
\left(1+\frac{y_t}{100}\right)^{1/12}-1
$$


#### Risk-free rate

In [70]:
df_RF = pd.read_csv("F-F_Research_Data_Factors.csv",
            usecols=["Unnamed: 0", "RF"],
           header=3,
           nrows=1201, 
           index_col=0
           )
df_RF.index = pd.to_datetime(df_RF.index, format="%Y"+"%m") + pd.offsets.MonthEnd(1)






In [71]:
df_RF["RF"] = (df_RF["RF"]/100)

In [72]:
df_RF["future_RF_1m"] = df_RF["RF"].shift(-1)  # Align the risk-free return with the next-month strategy return


In [73]:
backtest_df = pd.merge(
    analysis_1m_df,
    df_RF["future_RF_1m"],
    left_index=True,
    right_index=True,
    how="inner"
)

In [74]:
backtest_df["strategy_return"] = backtest_df["signal"]*backtest_df["future_return_1m"]+(1-backtest_df["signal"])*backtest_df["future_RF_1m"]


In [75]:
backtest_df["strategy_wealth"] = (1 + backtest_df["strategy_return"]).cumprod()

In [76]:
backtest_df["benchmark_wealth"] = (1 + backtest_df["future_return_1m"]).cumprod()  # Recompute for consistency


In [77]:
T_years_backtest = len(backtest_df)/12
T_years_backtest

37.5

In [78]:
backtest_strategy_cagr = ((backtest_df["strategy_wealth"].iloc[-1])**(1/T_years_backtest))-1
backtest_strategy_cagr

np.float64(0.11417940901645895)

In [79]:
backtest_benchmark_cagr = ((backtest_df["benchmark_wealth"].iloc[-1])**(1/T_years_backtest))-1
backtest_benchmark_cagr

np.float64(0.11235371471365663)

#### Volatility (annualized)


In [80]:
backtest_strategy_vol = backtest_df["strategy_return"].std()*np.sqrt(12)
backtest_benchmark_vol = backtest_df["future_return_1m"].std()*np.sqrt(12)

In [81]:
backtest_strategy_vol, backtest_benchmark_vol

(np.float64(0.11916290762500381), np.float64(0.1469157726840142))

#### Sharpe Ratio

$$
SR_{\text{annual}}
=
\sqrt{12}
\cdot
\frac{
\overline{R_p-R_f}
}{
\sigma(R_p-R_f)
}
$$

In [82]:
backtest_df["strategy_excess_return"] = backtest_df["strategy_return"] - backtest_df["future_RF_1m"]
backtest_df["benchmark_excess_return"] = backtest_df["future_return_1m"] - backtest_df["future_RF_1m"]

In [83]:
# Mean excess return
strategy_excess_return_mean = backtest_df["strategy_excess_return"].mean()
benchmark_excess_return_mean = backtest_df["benchmark_excess_return"].mean()


In [84]:
# Standard deviation of excess returns
strategy_excess_return_vol = backtest_df["strategy_excess_return"].std()
benchmark_excess_return_vol = backtest_df["benchmark_excess_return"].std()


In [85]:
# Monthly Sharpe ratio
strategy_sharpe_monthly = strategy_excess_return_mean/strategy_excess_return_vol
benchmark_sharpe_monthly = benchmark_excess_return_mean/benchmark_excess_return_vol

In [86]:
strategy_sharpe_annualy = strategy_sharpe_monthly*np.sqrt(12)
benchmark_sharpe_annualy = benchmark_sharpe_monthly*np.sqrt(12)

In [87]:
strategy_sharpe_annualy, benchmark_sharpe_annualy

(np.float64(0.7346339417683361), np.float64(0.6094773343322737))

#### Maximum Drawdown (MDD) — the largest peak-to-trough decline in portfolio wealth.


In [88]:
# Running wealth peaks
backtest_df["peak_strategy"] = backtest_df["strategy_wealth"].cummax()
backtest_df["peak_benchmark"] = backtest_df["benchmark_wealth"].cummax()


In [89]:
# Drawdown for each month
backtest_df["DD_strategy"] = (backtest_df["strategy_wealth"]/backtest_df["peak_strategy"])-1
backtest_df["DD_benchmark"] = (backtest_df["benchmark_wealth"]/backtest_df["peak_benchmark"])-1


In [90]:
MDD_strategy = backtest_df["DD_strategy"].min()
MDD_benchmark = backtest_df["DD_benchmark"].min()
MDD_strategy, MDD_benchmark

(np.float64(-0.19598015484016906), np.float64(-0.5094876760691069))

#### When did the maximum drawdown occur?


#### **methodological note:** Rows are indexed by decision date \(t\). Forward returns represent \(t \rightarrow t+1\); therefore realized wealth and drawdown stored on row \(t\) economically correspond to the end of \(t+1\).

#### Crisis periods


In [91]:
# 2008
backtest_df[["return_12m",
"signal",
"future_return_1m",
"future_RF_1m",
"strategy_return",
"strategy_wealth",
"DD_strategy"]].loc["2007-01-31":"2010-01-31"]

,return_12m,signal,future_return_1m,future_RF_1m,strategy_return,strategy_wealth,DD_strategy
2007-01-31,0.145134,1,-0.019561,0.0038,-0.019561,8.695922,-0.019561
2007-02-28,0.119695,1,0.011187,0.0043,0.011187,8.793202,-0.008593
2007-03-31,0.118300,1,0.044298,0.0044,0.044298,9.182719,0.000000
2007-04-30,0.152368,1,0.034893,0.0041,0.034893,9.503134,0.000000
2007-05-31,0.227917,1,-0.016612,0.0040,-0.016612,9.345265,-0.016612
2007-06-30,0.205886,1,-0.031006,0.0040,-0.031006,9.055504,-0.047103
2007-07-31,0.161328,1,0.014988,0.0042,0.014988,9.191232,-0.032821
2007-08-31,0.151340,1,0.037400,0.0032,0.037400,9.534987,0.000000
2007-09-30,0.164396,1,0.015907,0.0032,0.015907,9.686662,0.000000
2007-10-31,0.145591,1,-0.041808,0.0034,-0.041808,9.281678,-0.041808


In [92]:
# 2020
backtest_df[["return_12m",
"signal",
"future_return_1m",
"future_RF_1m",
"strategy_return",
"strategy_wealth",
"DD_strategy"]].loc["2019-02-28":"2021-02-28"]

,return_12m,signal,future_return_1m,future_RF_1m,strategy_return,strategy_wealth,DD_strategy
2019-02-28,0.046797,1,0.019431,0.0019,0.019431,21.908275,-0.114952
2019-03-31,0.094965,1,0.040489,0.0021,0.040489,22.795326,-0.079117
2019-04-30,0.134944,1,-0.063548,0.0021,-0.063548,21.346728,-0.137637
2019-05-31,0.037827,1,0.070477,0.0018,0.070477,22.851175,-0.076860
2019-06-30,0.104174,1,0.014373,0.0019,0.014373,23.179618,-0.063592
2019-07-31,0.079858,1,-0.015841,0.0016,-0.015841,22.812421,-0.078426
2019-08-31,0.029216,1,0.018711,0.0018,0.018711,23.239257,-0.061183
2019-09-30,0.042539,1,0.021659,0.0015,0.021659,23.742596,-0.040849
2019-10-31,0.143261,1,0.036299,0.0012,0.036299,24.604430,-0.006033
2019-11-30,0.161100,1,0.030183,0.0014,0.030183,25.347061,0.000000


### Robustness check across lookback windows


In [93]:
# Returns over alternative lookback horizons
analysis_1m_df["return_6m"] = analysis_1m_df["Close"] / analysis_1m_df["Close"].shift(6) - 1
analysis_1m_df["return_9m"] = analysis_1m_df["Close"] / analysis_1m_df["Close"].shift(9) - 1
analysis_1m_df["return_12m"] = analysis_1m_df["Close"] / analysis_1m_df["Close"].shift(12) - 1
analysis_1m_df["return_18m"] = analysis_1m_df["Close"] / analysis_1m_df["Close"].shift(18) - 1

analysis_1m_df[[
    "return_6m",
    "return_9m",
    "return_12m",
    "return_18m"
]].tail()


,return_6m,return_9m,return_12m,return_18m
Date,,,,
2026-02-28,0.071244,0.174316,0.169915,0.241353
2026-03-31,-0.017944,0.061843,0.178034,0.154875
2026-04-30,0.060275,0.147506,0.310529,0.287730
2026-05-31,0.113352,0.183903,0.297814,0.280350
2026-06-30,0.102075,0.131335,0.223250,0.299125


In [94]:
# Signals
analysis_1m_df["signal_6m"] = (analysis_1m_df["return_6m"] > 0).astype(int)
analysis_1m_df["signal_9m"] = (analysis_1m_df["return_9m"] > 0).astype(int)
analysis_1m_df["signal_12m"] = (analysis_1m_df["return_12m"] > 0).astype(int)
analysis_1m_df["signal_18m"] = (analysis_1m_df["return_18m"] > 0).astype(int)

analysis_1m_df[
[
"return_6m",
"signal_6m",
"return_9m",
"signal_9m",
"return_12m",
"signal_12m",
"return_18m",
"signal_18m"
]
].loc["2008-01-31":"2010-01-31"]


,return_6m,signal_6m,return_9m,signal_9m,return_12m,signal_12m,return_18m,signal_18m
Date,,,,,,,,
2008-01-31,-0.043187,0,-0.056442,0,-0.023112,0,0.111174,1
2008-02-29,-0.087937,0,-0.117872,0,-0.035987,0,0.050095,1
2008-03-31,-0.124616,0,-0.106845,0,-0.050770,0,0.019293,1
2008-04-30,-0.096358,0,-0.033375,0,-0.046767,0,0.035204,1
2008-05-31,-0.044714,0,-0.035313,0,-0.066975,0,0.029041,1
2008-06-30,-0.119138,0,-0.148488,0,-0.131201,0,-0.070746,0
2008-07-31,-0.070810,0,-0.168868,0,-0.110939,0,-0.092286,0
2008-08-31,-0.025721,0,-0.120056,0,-0.111397,0,-0.060783,0
2008-09-30,-0.108685,0,-0.192865,0,-0.219758,0,-0.153937,0


In [95]:
# Count signal switches; this will matter for transaction costs
for signal in ["signal_6m", "signal_9m", "signal_12m", "signal_18m"]:
    print(signal, analysis_1m_df[signal].diff().abs().sum())


signal_6m 53.0
signal_9m 37.0
signal_12m 21.0
signal_18m 17.0


#### Reusable backtest function


In [96]:
# Prepare a separate DataFrame for robustness checks
robustness_df = analysis_1m_df[
    [
        "future_return_1m",
        "signal_6m",
        "signal_9m",
        "signal_12m",
        "signal_18m"
    ]
].copy()


In [97]:
robustness_df = pd.merge(
    robustness_df,
    df_RF["future_RF_1m"],
    left_index=True,
    right_index=True,
    how="inner"
)

To compare lookback windows fairly, all strategies should use the same evaluation sample.


In [98]:
robustness_df = robustness_df.loc["1990-07-31":]

In [99]:
robustness_df[
    ["signal_6m", "signal_9m", "signal_12m", "signal_18m",
     "future_return_1m", "future_RF_1m"]
].isna().sum()


signal_6m           0
signal_9m           0
signal_12m          0
signal_18m          0
future_return_1m    0
future_RF_1m        0
dtype: int64

In [100]:
def run_backtest(df, signal_column):
    signal = df[signal_column]
    strategy_return = signal * df["future_return_1m"] + (1-signal) * df["future_RF_1m"]
    strategy_wealth = (1 + strategy_return).cumprod()
    benchmark_wealth = (1 + df["future_return_1m"]).cumprod()
    strategy_excess_return = strategy_return - df["future_RF_1m"]
    benchmark_excess_return = df["future_return_1m"] - df["future_RF_1m"]

    strategy_excess_return_mean = strategy_excess_return.mean()
    strategy_excess_return_std = strategy_excess_return.std()

    benchmark_excess_return_mean = benchmark_excess_return.mean()
    benchmark_excess_return_std = benchmark_excess_return.std()

    strategy_sharpe_monthly = strategy_excess_return_mean / strategy_excess_return_std
    benchmark_sharpe_monthly = benchmark_excess_return_mean / benchmark_excess_return_std

    strategy_sharpe_annual = strategy_sharpe_monthly*np.sqrt(12)
    benchmark_sharpe_annual = benchmark_sharpe_monthly*np.sqrt(12)

    T_years = len(df) / 12

    strategy_cagr = ((strategy_wealth.iloc[-1])**(1/T_years)) - 1
    benchmark_cagr = ((benchmark_wealth.iloc[-1])**(1/T_years)) - 1

    strategy_vol_annual = strategy_return.std() * np.sqrt(12)
    benchmark_vol_annual = df["future_return_1m"].std() * np.sqrt(12)

    strategy_peak = strategy_wealth.cummax()
    benchmark_peak = benchmark_wealth.cummax()

    # Percentage drawdowns
    strategy_drawdown = strategy_wealth / strategy_peak - 1
    benchmark_drawdown = benchmark_wealth / benchmark_peak - 1

    strategy_mdd = strategy_drawdown.min()
    benchmark_mdd = benchmark_drawdown.min()
    return {
        "lookback": signal_column,
        "strategy_cagr": strategy_cagr,
        "benchmark_cagr": benchmark_cagr,
        "strategy_vol_annual": strategy_vol_annual,
        "benchmark_vol_annual": benchmark_vol_annual,
        "strategy_sharpe_annual": strategy_sharpe_annual,
        "benchmark_sharpe_annual": benchmark_sharpe_annual,
        "strategy_mdd": strategy_mdd,
        "benchmark_mdd": benchmark_mdd
    }


In [101]:
result_12m = run_backtest(robustness_df, "signal_12m")
result_12m

{'lookback': 'signal_12m',
 'strategy_cagr': np.float64(0.11202534668552411),
 'benchmark_cagr': np.float64(0.11012732333168507),
 'strategy_vol_annual': np.float64(0.11836683268278171),
 'benchmark_vol_annual': np.float64(0.14732374288995048),
 'strategy_sharpe_annual': np.float64(0.7401183563273638),
 'benchmark_sharpe_annual': np.float64(0.608911309502056),
 'strategy_mdd': np.float64(-0.19598015484016895),
 'benchmark_mdd': np.float64(-0.5094876760691069)}

In [102]:
signals = ["signal_6m", "signal_9m", "signal_12m", "signal_18m"]
results = []
for elem in signals:
    results.append(run_backtest(robustness_df, elem))

pd.DataFrame(results)

,lookback,strategy_cagr,benchmark_cagr,strategy_vol_annual,benchmark_vol_annual,strategy_sharpe_annual,benchmark_sharpe_annual,strategy_mdd,benchmark_mdd
0,signal_6m,0.094050,0.110127,0.109399,0.147324,0.641227,0.608911,-0.220721,-0.509488
1,signal_9m,0.104493,0.110127,0.114953,0.147324,0.699939,0.608911,-0.220670,-0.509488
2,signal_12m,0.112025,0.110127,0.118367,0.147324,0.740118,0.608911,-0.195980,-0.509488
3,signal_18m,0.101262,0.110127,0.123042,0.147324,0.637241,0.608911,-0.261442,-0.509488


#### The 12-month lookback was pre-specified and performs best in this comparison.


### **Transaction costs / implementation realism**